In [37]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel ,Field

In [38]:
llm = ChatOllama(
    model = "qwen2.5-coder:7b",
    temperature=0.9
)

In [39]:
result = llm.invoke("hi")
print(result)

content="Hello! How can I assist you today? Is there anything specific you would like to know or discuss? I'm here to help with any questions you may have." additional_kwargs={} response_metadata={'model': 'qwen2.5-coder:7b', 'created_at': '2026-02-07T19:15:51.343784Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14520940917, 'load_duration': 5288655042, 'prompt_eval_count': 30, 'prompt_eval_duration': 5312290666, 'eval_count': 34, 'eval_duration': 3121603790, 'logprobs': None, 'model_name': 'qwen2.5-coder:7b', 'model_provider': 'ollama'} id='lc_run--019c3987-f2f3-77f0-948f-253abfb5e5b6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 30, 'output_tokens': 34, 'total_tokens': 64}


In [40]:
class Person(BaseModel):
    name : str = Field(description="Name of the person.")
    age : int = Field(gt=18, description="Age of that person ")
    city : str = Field(description="Name of the city from which the person belong")


In [33]:
parser = PydanticOutputParser(pydantic_object=Person)
temmplate1 = PromptTemplate(
    template="Give the description about the president of {place} like name,age and city \n {format_instruction}",
    input_variables=["place"],
    partial_variables={"format_instruction" : parser.get_format_instructions()}
)

In [41]:
prompt = temmplate1.invoke({'place':'India'})
print(prompt)

text='Give the description about the president of India like name,age and city \n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "Name of the person.", "title": "Name", "type": "string"}, "age": {"description": "Age of that person ", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "Name of the city from which the person belong", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```'


In [42]:
result = llm.invoke(prompt)
print(result)

content='Here is a well-formatted instance of the schema with an example for the president of India:\n\n```json\n{\n  "name": "Narendra Modi",\n  "age": 70,\n  "city": "Ahmedabad"\n}\n```\n\nThis instance includes the name, age, and city of the current president of India, Narendra Modi.' additional_kwargs={} response_metadata={'model': 'qwen2.5-coder:7b', 'created_at': '2026-02-07T19:16:14.051667Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10374238958, 'load_duration': 114511291, 'prompt_eval_count': 263, 'prompt_eval_duration': 2862868917, 'eval_count': 73, 'eval_duration': 5620235330, 'logprobs': None, 'model_name': 'qwen2.5-coder:7b', 'model_provider': 'ollama'} id='lc_run--019c3988-5bd2-7b62-817d-2e599ced3d8f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 263, 'output_tokens': 73, 'total_tokens': 336}


In [43]:
final_result = parser.parse(result.content)
print(final_result)

name='Narendra Modi' age=70 city='Ahmedabad'


In [45]:
chain = temmplate1 | llm |parser

result = chain.invoke({'place':'India'})
print(result)

name='Narendra Modi' age=78 city='Gujarat'
